In [1]:
import os
import sys
from osgeo import gdal
import re
#import rasterio
import numpy as np
import pandas as pd
import odc.stac
import duckdb
import fsspec
from shapely.geometry import box
import pystac
from stac_geoparquet.arrow._api import stac_table_to_items


def get_query(start_date, end_date, wkt_bbox, stac_geoparquet):
    """Generate SQL query for filtering STAC items by date and geometry."""
    sql_where = f'(("datetime" BETWEEN \'{start_date}T00:00:00Z\' AND \'{end_date}T23:59:59Z\')) AND (ST_Intersects(geometry, ST_GeomFromText(\'{wkt_bbox}\')) )'
    sql_query = f"SELECT * EXCLUDE(geometry),ST_AsWKB(geometry) as geometry FROM read_parquet('{stac_geoparquet}', union_by_name=False) WHERE {sql_where}"
    return sql_query


def add_sas_token(item, sas_token):
    """Add SAS token to all asset URLs in a STAC item."""
    for _, asset in item.assets.items():
        asset.href = f"{asset.href}?{sas_token}"
    return item


def is_leap_year(year):
    return (year % 4 == 0) and (year % 100 != 0 or year % 400 == 0)


DAYCOUNT_LEAP = [31,29,31,30,31,30,31,31,30,31,30,31]
DAYCOUNT_NOLEAP = [31,28,31,30,31,30,31,31,30,31,30,31]

YEAR = 2019
MONTH = 8


# Define bounding box and time frame for Germany for S3 compositing
bbox = [5.592041, 47.129951, 15.26001, 55.09723]  # Germany
bbox_shape = box(*bbox)
wkt_bbox = bbox_shape.wkt

start = f"{YEAR}-{MONTH:02d}-01"
if is_leap_year(YEAR):
   end = f"{YEAR}-{MONTH:02d}-{DAYCOUNT_LEAP[MONTH-1]}"
else:
    end = f"{YEAR}-{MONTH:02d}-{DAYCOUNT_NOLEAP[MONTH-1]}"

# read AZURE CLOUD KEY from file in personal folder
with open ('/data/Aldhani/users/potzschf/rss.txt') as f:
    STORAGE_ACCOUNT = f.readline().strip()
    SAS_KEY = f.readline().strip()


# Setup Azure filesystem and DuckDB connection
azure_blob_fileservice = fsspec.filesystem("abfs", account_name=STORAGE_ACCOUNT, sas_token=SAS_KEY)
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.register_filesystem(azure_blob_fileservice)
stac_geoparquet = "abfs://lst/catalog.parquet"

# Execute query
sql_query = get_query(start, end, wkt_bbox, stac_geoparquet)
db = con.query(sql_query)
table = db.to_arrow_table() # fetch_arrow depreciated

# Convert to STAC items and add SAS tokens
items = [
    add_sas_token(pystac.Item.from_dict(item_dict), SAS_KEY)
    for item_dict in stac_table_to_items(table)
]

In [10]:
for i, item_dict in enumerate(stac_table_to_items(table)):
    assets = item_dict.get("assets", {})

    bad_assets = [
        key for key, value in assets.items()
        if value is None
    ]

    if bad_assets:
        print(f"Item {i} has invalid assets: {bad_assets}")
     

Item 0 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 1 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 2 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 3 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 4 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 5 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 6 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 7 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 8 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 9 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 10 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 11 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 12 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 13 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 14 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 15 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 16 has invalid assets: ['sat_azimuth', 'sat_zenith']
Item 17 has invalid asse

In [ ]:
import requests



614.0 / 614.0 MB
Download complete: crop_type_map_2020.tif


In [20]:
[asset for asset in item["assets"].values() if "tiff" in asset.get("type", "").lower()]


[{'href': 'https://eodata.thuenen.de/cog/crop_type_map_v302/CTM_GER_2020_rst_v302_COG.tif',
  'type': 'image/tiff; application=geotiff; profile=cloud-optimized',
  'roles': ['data'],
  'title': 'Crop Type Map 2020',
  'description': 'Annual 10 m agricultural land use classification maps for Germany based on [v302](https://eodata.thuenen.de/collections/crop-type-map-v302), combining finalized and preliminary in-season products into a continuously updated time series (2017-2025).\n\nThis dynamic dataset provides the longest and most up-to-date timeline of agricultural land use maps. It includes final maps for all available years as well as preliminary in-season products for the most recent year, which are released once sufficient input data allow reliable predictions, typically in autumn. Preliminary maps may be updated as additional observations become available. Once a year is finalized, the corresponding map is also released as a persistent [v302](https://eodata.thuenen.de/collections